# RES-138 — Voyage 4 Nano vs Qwen3 Embedding 0.6B (Colab preflight)

**This notebook orchestrates; it does not implement.** Every algorithm — the BEIR
loader, the MRL derivation, the calibration gate, the artifact contracts, the preflight
bundle — lives in `dynamisrag.benchmark` in the repository and is unit tested there.
Nothing here parses a corpus, computes a metric, normalises a vector or decides anything
about a model.

**What this session does:** clone an exact commit, prove the environment, verify the
three frozen BEIR archives, verify the pinned model prompts and pooling, calibrate the
Matryoshka 512 derivation on a fixed sample, write one preflight artifact, print its
SHA-256, and stop.

**What this session does not do:** embed any corpus, compute any retrieval metric,
select a model, encode a default, or run Docker. A full corpus pass is unreachable here
without an approved preflight digest, and it is not implemented in this tranche.

**Run mode:** `RUN_MODE = "preflight"`. The second-to-last cell stops; the last cell
refuses to embed anything.

## 0. Parameters

In [ ]:
# --- Parameters. Paste the exact 40-character commit reported for the RES-138 harness branch.
REPO_URL = "https://github.com/Litju/DynamisRAG.git"
CODE_SHA = ""

RUN_MODE = "preflight"
APPROVED_PREFLIGHT_SHA256 = ""

DRIVE_ROOT = "/content/drive/MyDrive/DynamisRAG/RES-138"

SHARD_SIZE = 4096

CANDIDATE_DIMENSIONS = (512, 1024)

BOOTSTRAP_SEED = 138
BOOTSTRAP_SAMPLES = 10_000
BOOTSTRAP_CONFIDENCE = 0.95

# Frozen identities, repeated here so the notebook states what it is about to check
# without a second source of truth. The repository declares these and is authoritative;
# the assertions in step 8 and step 9 fail if the two ever disagree.
BEIR_SOURCES = {
    "scifact": "536e14446a0ba56ed1398ab1055f39fe852686ecad24a6306c80c490fa8e0165",
    "nfcorpus": "efe5be03f8c5b86a5870102d0599d227c8c6e2484328e68c6522560385671b0b",
    "trec-covid": "120f42a7864d2214234537733c0d2c6684e42fdfafff2c5eacf98afca6656aa0",
}

MODEL_CANDIDATES = {
    "voyageai/voyage-4-nano": "67fabc9bef010dabc5f6024aa1b1b6b93410426f",
    "Qwen/Qwen3-Embedding-0.6B": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3",
}

## 1. GPU validation

Refused here rather than twenty minutes into a model load. A CPU-only runtime is not a
reduced version of this benchmark: the measured throughput on a CPU host was
0.079–0.215 documents per second, which makes `Recall@100` identical for every candidate
and therefore useless as a discriminator.

In [ ]:
import sys

import torch

from dynamisrag.benchmark.runtime import require_cuda_available

require_cuda_available(
    available=bool(torch.cuda.is_available()),
    device_count=int(torch.cuda.device_count()),
    operation="notebook_gpu_validation",
)
print(f"CUDA available: {torch.cuda.is_available()}  devices: {torch.cuda.device_count()}")
print(f"torch {torch.__version__}  cuda {torch.version.cuda}")

## 2. Mount Drive and check the storage contract

Paths, not folder ids: the ids are recorded for provenance and documentation, and a run
that needed one in order to start would break the first time the tree is reorganised. All
active work happens under `/content/res138`; Drive holds verified archives and finished
artifacts only, and is never written a matrix byte by byte.

In [ ]:
from pathlib import Path

from google.colab import drive

from dynamisrag.benchmark.contracts import RES138_DRIVE_LOCATIONS
from dynamisrag.benchmark.res138 import Res138ColabConfig

config = Res138ColabConfig(
    code_sha=CODE_SHA,
    run_mode=RUN_MODE,
    approved_preflight_sha256=APPROVED_PREFLIGHT_SHA256,
    shard_size=SHARD_SIZE,
    candidate_dimensions=CANDIDATE_DIMENSIONS,
    bootstrap_seed=BOOTSTRAP_SEED,
    bootstrap_samples=BOOTSTRAP_SAMPLES,
    bootstrap_confidence=BOOTSTRAP_CONFIDENCE,
    drive_root=DRIVE_ROOT,
)

drive.mount("/content/drive", force_remount=False)

SCRATCH = Path("/content/res138")
RUNS_ROOT = Path(config.runs_path)
BEIR_CACHE = Path(config.beir_cache_path)
REPO_DIR = SCRATCH / "repo"
HF_CACHE = SCRATCH / "hf-cache"

for directory in (SCRATCH, RUNS_ROOT, BEIR_CACHE, HF_CACHE):
    directory.mkdir(parents=True, exist_ok=True)

missing = [
    location.path
    for location in RES138_DRIVE_LOCATIONS
    if not Path(location.path).is_dir()
]
if missing:
    raise RuntimeError(
        f"these Drive folders do not exist: {missing}. Create them once, or check that "
        f"DRIVE_ROOT is {DRIVE_ROOT!r}."
    )
print(f"scratch     {SCRATCH}")
print(f"runs        {RUNS_ROOT}")
print(f"beir cache  {BEIR_CACHE}")

## 3. Clone the exact commit

GitHub is the only code transport — no bundle, no uploaded archive. The commit is checked
out detached and then *verified*: `git rev-parse HEAD` must equal `CODE_SHA` and the tree
must be clean. Colab never commits and never pushes.

In [ ]:
import subprocess
from pathlib import Path


def git(*arguments: str, cwd: Path | None = None) -> str:
    """Run one git command, refusing a non-zero exit rather than printing its noise."""
    completed = subprocess.run(
        ["git", *arguments],
        cwd=str(cwd) if cwd is not None else None,
        capture_output=True,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(arguments)} failed ({completed.returncode}): {completed.stderr.strip()[:400]}"
        )
    return completed.stdout.strip()


if not CODE_SHA:
    raise RuntimeError(
        "CODE_SHA is empty. Paste the exact 40-character commit reported for the RES-138 "
        "harness branch: a branch name or a tag resolves to a commit that is not known until the "
        "clone runs, so it cannot bind this run to its code."
    )
git("clone", "--no-checkout", REPO_URL, str(REPO_DIR))
git("fetch", "--depth", "1", "origin", CODE_SHA, cwd=REPO_DIR)
git("checkout", "--detach", CODE_SHA, cwd=REPO_DIR)

head = git("rev-parse", "HEAD", cwd=REPO_DIR)
if head != CODE_SHA:
    raise RuntimeError(f"checked out {head!r} but CODE_SHA is {CODE_SHA!r}")
dirty = git("status", "--porcelain", cwd=REPO_DIR)
if dirty:
    raise RuntimeError(f"the checkout is not clean:\n{dirty[:500]}")

sys.path.insert(0, str(REPO_DIR / "src"))
import dynamisrag  # noqa: E402

print(f"dynamisrag from {dynamisrag.__file__}")
print(f"HEAD {head}")

## 4. Install the pinned benchmark dependencies

`requirements/res138-colab.txt` pins no torch on purpose: Colab owns the CUDA build. The
next cell proves that installing it changed nothing about the runtime.

In [ ]:
from importlib.metadata import PackageNotFoundError, version

TORCH_BEFORE = {
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda or "none"),
    "torch_cuda_build": str(torch.version.cuda or "none"),
}


def installed_version(name: str) -> str:
    """The installed version of one distribution, or a marker if it is absent."""
    try:
        return version(name)
    except PackageNotFoundError:
        return "not-installed"


LIBRARY_NAMES = ("sentence-transformers", "transformers", "huggingface-hub", "numpy", "torch")
LIBRARIES_BEFORE = {name: installed_version(name) for name in LIBRARY_NAMES}

completed = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--requirement",
        str(REPO_DIR / "requirements" / "res138-colab.txt"),
    ],
    capture_output=True,
    text=True,
    check=False,
)
if completed.returncode != 0:
    raise RuntimeError(f"pip install failed ({completed.returncode}): {completed.stderr[-1500:]}")

for name in LIBRARY_NAMES:
    print(f"  {name}: {LIBRARIES_BEFORE.get(name)} -> {installed_version(name)}")

## 5. Prove the install did not replace PyTorch or CUDA

A requirements file that swaps the CUDA build changes the numerics of a run that has
already recorded its fingerprint, so the comparison is checked rather than assumed.

In [ ]:
from dynamisrag.benchmark.runtime import require_torch_unchanged

TORCH_AFTER = {
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda or "none"),
    "torch_cuda_build": str(torch.version.cuda or "none"),
}
require_torch_unchanged(
    before=TORCH_BEFORE, after=TORCH_AFTER, operation="notebook_dependency_install"
)
for name, value in sorted(TORCH_AFTER.items()):
    print(f"{name}: {value}")

## 6. Capture the runtime fingerprint

Python, platform, GPU name, VRAM, compute capability, NVIDIA driver, CUDA runtime, and
the resolved versions of torch, numpy, sentence-transformers, transformers and
huggingface-hub, hashed into `res138-runtime-v1`. The run id is *derived* from the code
commit, the GPU and that digest, so an interrupted session reopens the same folder and a
different card lands in a different one.

In [ ]:
from dynamisrag.benchmark.runner import probe_colab_runtime
from dynamisrag.benchmark.runtime import capture_runtime_fingerprint

driver = subprocess.run(
    ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=False,
)
if driver.returncode != 0:
    raise RuntimeError(
        f"nvidia-smi could not report the driver version ({driver.returncode}): "
        f"{driver.stderr.strip()[:300]}. The runtime fingerprint requires it; it is the field "
        "that changes most often when a session reconnects onto different hardware."
    )

fingerprint = capture_runtime_fingerprint(
    probe_colab_runtime(
        code_sha=CODE_SHA, nvidia_driver_version=driver.stdout.strip().splitlines()[0]
    )
)
RUNTIME_SHA = fingerprint.sha256
RUN_ID = fingerprint.run_id

print(f"run id   {RUN_ID}")
print(f"runtime  {RUNTIME_SHA}")
for key, value in sorted(dict(fingerprint.payload).items()):
    print(f"  {key}: {value}")

## 7. Write the frozen plan and create the run directory

`run-001` is never pre-created: the folder is named by the run id and carries a manifest
binding the code commit, the runtime fingerprint, the plan digest, the generation
semantics, the shard revision and size, the candidate model revisions and the dataset
digests. Re-opening is refused unless all of them match exactly — a partially resumed run
under two identities is worse than starting again.

The dataset digests come from the frozen specs rather than from a download, because they
are frozen *before* the download; step 8 proves the archives on disk are those bytes.

In [ ]:
from dynamisrag.benchmark.contracts import RES138_BEIR_SOURCES
from dynamisrag.benchmark.res138 import benchmark_plan, create_res138_run, generation_semantics_sha256

plan = benchmark_plan(CODE_SHA)
PLAN_SHA = plan.sha256
plan.write(RUNS_ROOT / "res138-plan.json")

RUN_DIRECTORY, run_manifest = create_res138_run(
    runs_root=RUNS_ROOT,
    config=config,
    fingerprint=fingerprint,
    dataset_digests=[(source.workload, source.sha256) for source in RES138_BEIR_SOURCES],
)
GENERATION_SEMANTICS_SHA = generation_semantics_sha256()

print(f"plan       {PLAN_SHA}")
print(f"run        {RUN_DIRECTORY}")
print(f"identity   {run_manifest.identity_sha256}")
print(f"semantics  {GENERATION_SEMANTICS_SHA}")

## 8. Verify, cache and load the frozen BEIR archives

Download to `/content/res138`, verify the SHA-256 **before** extraction, copy the
verified archive into the Drive cache and re-verify the copy. A cached archive is
re-verified on every use: Drive is a synchronising network filesystem shared with a
browser, and the cost of the check is one hash of a file that is about to be read anyway.

In [ ]:
from dynamisrag.benchmark.res138 import verify_and_cache_beir_sources

loaded = verify_and_cache_beir_sources(
    scratch_dir=SCRATCH / "sources",
    cache_dir=BEIR_CACHE,
    extract_dir=SCRATCH / "extracted",
)

for entry in loaded:
    summary = entry.workload.summary()
    print(
        f"{summary['name']:<11} zip {entry.source.sha256[:16]}  "
        f"documents {summary['document_count']:>6}  queries {summary['query_count']:>4}  "
        f"qrels {summary['qrel_count']:>6}  excluded-no-text "
        f"{entry.report.documents_without_embedding_text}"
    )

assert {entry.workload.name: entry.source.sha256 for entry in loaded} == BEIR_SOURCES, (
    "the archives on disk are not the frozen ones"
)

## 9. Verify the pinned model metadata

At each candidate's exact revision: the model-native query and document prompts must be
the frozen ones byte for byte, the pooling mode must be the frozen one, the similarity
function must be cosine, and the model must carry its own normalisation stage. A
mismatch fails the run — nothing is substituted, because a changed prompt changes the
vectors and would invalidate the calibration and any result computed under it.

In [ ]:
from dynamisrag.benchmark.res138 import verify_pinned_model_metadata
from dynamisrag.benchmark.runner import HubModelMetadataReader

model_provenance = verify_pinned_model_metadata(HubModelMetadataReader())

for entry in model_provenance:
    print(f"{entry['model_id']}@{entry['revision'][:12]}  pooling={entry['pooling_mode']}")
    print(f"  prompt pair sha256 {entry['prompt_sha256']}")

assert {
    entry["model_id"]: entry["revision"] for entry in model_provenance
} == MODEL_CANDIDATES

## 10. Calibrate the Matryoshka 512 derivation

One 1024-dimensional pass per item plus one native 512 pass, for both models and both
paths — the query path and the document path decided separately, because the two models
pool differently (mean versus last token) and the two paths are prompted differently. The
gate was predeclared before this ran: worst cosine at least 0.999999, worst absolute
component difference at most 1e-5, identical calibration top-10 ordering. It is not
adjusted after seeing the numbers; a failing pair gets a native 512 pass in the full run.

The calibration sample is chosen by a rule in the repository — within-workload length
thirds, then the smallest SHA-256 of the identity, two items per cell — not by hand.

In [ ]:
from dynamisrag.benchmark.calibration import select_calibration_set
from dynamisrag.benchmark.contracts import RES138_MODEL_CANDIDATES
from dynamisrag.benchmark.res138 import run_mrl_calibration
from dynamisrag.benchmark.runner import SentenceTransformersCalibrationEncoder

CALIBRATION_BATCH = 16
calibration_item_payloads = []
decisions = []
runner_provenance = []

for entry in loaded:
    calibration = select_calibration_set([entry.workload])
    calibration_item_payloads.extend(item.payload() for item in calibration.items)
    for candidate in RES138_MODEL_CANDIDATES:
        encoder = SentenceTransformersCalibrationEncoder(
            candidate=candidate,
            batch_size=CALIBRATION_BATCH,
            cache_folder=HF_CACHE,
        )
        runner_provenance.append({**encoder.describe(), "workload": entry.workload.name})
        decisions.extend(run_mrl_calibration(encoder=encoder, calibration=calibration))

for decision in decisions:
    suffix = f"  failed: {list(decision.failures())}" if decision.failures() else ""
    print(
        f"{decision.model_id}@{decision.model_revision[:12]} {decision.kind.value:<9} "
        f"cos>={decision.minimum_cosine:.9f}  "
        f"maxdiff={decision.maximum_absolute_difference:.2e}  "
        f"top-{decision.top_k} identical={decision.identical_top_k}  "
        f"derived512_allowed={decision.derived512_allowed}{suffix}"
    )

print(f"\ncalibration items: {len(calibration_item_payloads)}")
print(f"decisions: {len(decisions)}")

## 11. Write the preflight bundle

One artifact stating everything a full run would rely on: the code commit, the runtime
payload and its digest, the Drive run id, each archive's verified digest and what loading
it declared, each candidate's revision and prompt digests, the generation semantics, the
exact calibration inputs, the native-512-versus-derived-512 numbers, the per-model-per-path
MRL decision, and the digests of the artifacts already written. It does not authorise
itself — the authorisation is a human copying its digest into the parameter cell.

In [ ]:
from dynamisrag.benchmark.calibration import CalibrationItem, CalibrationSet
from dynamisrag.benchmark.res138 import PREFLIGHT_FILENAME, write_preflight_bundle

# Rebuild the calibration set from the payloads recorded above, so the artifact names
# the same items the decisions were made on rather than a second selection.
calibration_items = tuple(
    CalibrationItem(
        workload=str(payload["workload"]),
        kind=str(payload["kind"]),
        band=str(payload["band"]),
        item_id=str(payload["item_id"]),
        content_sha256=str(payload["content_sha256"]),
        length=int(payload["length"]),
        text="",
    )
    for payload in calibration_item_payloads
)

preflight_path = RUN_DIRECTORY / PREFLIGHT_FILENAME
PREFLIGHT_SHA = write_preflight_bundle(
    preflight_path,
    config=config,
    fingerprint=fingerprint,
    run_id=RUN_ID,
    loaded=loaded,
    model_provenance=model_provenance,
    calibration=CalibrationSet(items=calibration_items),
    decisions=decisions,
    artifact_digests={"res138-plan.json": PLAN_SHA},
)
print(f"wrote {preflight_path}")

## 12. Print the preflight SHA-256

This is the value to review and approve. It is the digest of the artifact on disk, which
is what the full run will compare `APPROVED_PREFLIGHT_SHA256` against.

In [ ]:
from dynamisrag.benchmark.res138 import preflight_digest, verify_preflight_bundle

verify_preflight_bundle(preflight_path, expect_code_sha=CODE_SHA, expect_run_id=RUN_ID)

print()
print("=" * 78)
print(f"PREFLIGHT PATH  {preflight_path}")
print(f"PREFLIGHT SHA   {PREFLIGHT_SHA}")
print(f"file digest     {preflight_digest(preflight_path)}")
print(f"run id          {RUN_ID}")
print(f"code sha        {CODE_SHA}")
print(f"plan sha        {PLAN_SHA}")
print("=" * 78)
print()
print("Review the artifact, then re-run with:")
print('    APPROVED_PREFLIGHT_SHA256 = "' + PREFLIGHT_SHA + '"')
print('    RUN_MODE = "full"')

## 13. Hard stop

Nothing after this line runs in preflight mode.

In [ ]:
config.require_preflight_mode(operation="notebook_hard_stop")

print("PREFLIGHT COMPLETE. No corpus was embedded, no metric was computed and no default was")
print("selected. Return the preflight SHA above for review.")

## Full run — unreachable without an approved preflight digest

Two conditions, both required: `RUN_MODE == "full"`, **and** the preflight artifact on
disk hashing to exactly `APPROVED_PREFLIGHT_SHA256`. The digest check is equality, not a
prefix: a re-written preflight — a changed driver, a different prompt, a failing MRL
decision — must not inherit an approval given to an earlier one.

This tranche implements the gate, not the pass. The cell below proves the gate and then
stops, so running this notebook cannot begin a corpus pass.

In [ ]:
if RUN_MODE != "full":
    print('RUN_MODE is not "full". The full corpus pass is not reachable from this session.')
else:
    from dynamisrag.benchmark.res138 import require_approved_preflight

    approved = require_approved_preflight(
        config=config, path=Path(config.runs_path) / RUN_ID / PREFLIGHT_FILENAME
    )
    print(f"approved preflight {approved.sha256}")
    raise SystemExit(
        "The full SciFact/NFCorpus/TREC-COVID embedding pass is NOT implemented in this tranche. "
        "The harness, the frozen contracts, the preflight gate and the bundle verifier are in "
        "place; the embedding, retrieval, metrics, results and selection artifacts land in a "
        "separate, separately reviewed commit after this preflight has been reviewed."
    )